In [0]:
path_txt = '/Volumes/sakshi/dataframe/raw/new 4.txt'
df_txt = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", "\t") \
    .load(path_txt)

output_csv = '/Volumes/sakshi/dataframe/raw/new_4.csv'
df_txt.write.mode("overwrite").option("header", "true").csv(output_csv)
display(df_txt)

In [0]:
# Parse the DataFrame correctly first - use comma delimiter instead of tab
path_txt = '/Volumes/sakshi/dataframe/raw/new 4.txt'
df_cleaned = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ",") \
    .load(path_txt)

# Drop the 'vehicle' column
df_cleaned = df_cleaned.drop("vehicle")

# Convert city column to lowercase
from pyspark.sql.functions import col, lower, to_date, coalesce, lit, when

df_cleaned = df_cleaned.withColumn("city", lower(col("city")))

# Add datetime function - convert opening_date to proper date type
df_cleaned = df_cleaned.withColumn("opening_date", to_date(col("opening_date"), "yyyy-MM-dd"))

# Handle null/empty values for all columns at once using withColumns
string_cols = [field.name for field in df_cleaned.schema.fields if str(field.dataType) == "StringType"]
numeric_cols = [field.name for field in df_cleaned.schema.fields if str(field.dataType) in ["IntegerType", "DoubleType", "LongType", "FloatType"]]

# Build dictionary for withColumns
update_dict = {}
for col_name in string_cols:
    update_dict[col_name] = when(col(col_name).isNull() | (col(col_name) == ""), lit(None)).otherwise(col(col_name))
for col_name in numeric_cols:
    update_dict[col_name] = coalesce(col(col_name), lit(0))

for col_name, col_expr in update_dict.items():
    df_cleaned = df_cleaned.withColumn(col_name, col_expr)

# Remove duplicates
df_cleaned = df_cleaned.dropDuplicates()

# Order by customer_id ascending
df_cleaned = df_cleaned.orderBy(col("customer_id").asc())

display(df_cleaned)

In [0]:
# Remove duplicates based on all columns
df_cleaned = df_cleaned.dropDuplicates(df_cleaned.columns)

display(df_cleaned)

In [0]:
from pyspark.sql.functions import current_date

df_cleaned = df_cleaned.withColumn("transformations", current_date())

display(df_cleaned)

In [0]:
df_cleaned = df_cleaned.drop("currnet_date")

display(df_cleaned)

In [0]:
from pyspark.sql.functions import datediff, current_date, format_string

df_cleaned = df_cleaned.withColumn("date_difference", datediff(current_date(), col("opening_date")))
df_cleaned = df_cleaned.withColumn("date_difference", format_string("%d days", col("date_difference")))

df_cleaned = df_cleaned.drop("transformations")

display(df_cleaned)